# STGAN — ConvGRU su griglia + LSTM del trend

Questo notebook **addestra un nuovo detector**. Mantiene LSTM 168 h, calendario,
loss e score STGAN; sostituisce le convoluzioni sul grafo nei gate GRU con Conv2d.
Generatore e discriminatore mantengono la ricorrenza nel ramo recente.
Il default `recent_steps=1` resta quello della configurazione di riferimento;
sono supportate sequenze maggiori di 1 senza cambiare il ramo LSTM.
`cnn_channels` e `cnn_layers` configurano canali di stato e strati ConvGRU.
Input: tre variabili su una finestra geografica più maschera delle celle presenti.
La baseline resta in `feat/stgan-paper`; gli output CNN hanno una directory propria.

Eseguire sul server che contiene il manifest preparato. Le analisi di previsione
rimangono posthoc e usano gli stessi CSV SDE-Net. Nessuna soglia viene ottimizzata
sul test: il top-1% è il budget globale di ranking predefinito.

In [ ]:
from pathlib import Path
import os
import sys
import json
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.anomaly_detection.stgan import STGANCNNConfig, STGAN, build_spatial_grid
from scripts.run_pvgis_stgan import run_stgan

manifest_candidates = [
    ROOT / 'outputs/pvgis_stgan/prepared/manifest.csv',
    ROOT / 'outputs/pvgis_stgan/prepared/manifest_shard_0000.csv',
    ROOT / 'outputs/pvgis_stgan_cnn/prepared/manifest.csv',
]
MANIFEST = Path(os.environ.get('STGAN_MANIFEST', str(next(
    (p for p in manifest_candidates if p.is_file()), manifest_candidates[0])))).resolve()
BASELINE_SEED_DIR = Path(os.environ.get('STGAN_BASELINE_SEED_DIR',
    str(ROOT / 'outputs/pvgis_stgan/paper_reference/seed_20'))).resolve()
SEEDS = (20,)
SEED = SEEDS[0]
DEVICE = os.environ.get('STGAN_DEVICE', 'cuda')
TOP_K_PERCENT = 1.0
# Run server: ablation kernel 5x5 con pipeline ottimizzata.
KERNEL_SIZE = 5  # Kernel convoluzionale; la patch geografica resta 3x3.
CONFIG = STGANCNNConfig(
    epochs=6, batch_size=256, hidden_size=64, n_layers=2,
    cnn_channels=32, cnn_layers=2, patch_size=3, kernel_size=KERNEL_SIZE,
    recent_steps=1, trend_steps=168, train_samples_per_epoch=0,
    train_num_workers=4, score_num_workers=4,
    persistent_workers=True, prefetch_factor=2, pin_memory=True,
    score_batch_size=1024, execution_mode='optimized',
    grid_crs=os.environ.get('STGAN_GRID_CRS', 'EPSG:32632'),
)
# Directory dedicata; STGAN_CNN_OUT_DIR, se impostata sul server, ha precedenza.
RUN_NAME = f'convgru_patch{CONFIG.patch_size}_kernel{CONFIG.kernel_size}_optimized'
OUT_ROOT = Path(os.environ.get('STGAN_CNN_OUT_DIR',
    str(ROOT / 'outputs/pvgis_stgan_cnn' / RUN_NAME))).resolve()
SEED_DIR = OUT_ROOT / f'seed_{SEED}'
RUN_TRAINING = True  # False: consultazione di un run già completato
print('Manifest:', MANIFEST)
print('Output CNN:', OUT_ROOT)
display(pd.Series(CONFIG.to_dict(), name='Configurazione'))

## Verifica della disposizione geografica

Il codice verifica una griglia regolare e assegna le celle in ordine geografico.
Non riordina la lista kNN come se fosse un'immagine. Per i blocchi completi
controlla anche se l'insieme dei punti coincide con quello dei vicini originali.
I conteggi 917/232 vanno confermati sul manifest reale.
Le celle assenti rimangono mascherate; la località centrale è sempre presente.

In [ ]:
manifest = pd.read_csv(MANIFEST, dtype={'location': str})
grid = build_spatial_grid(manifest.latitude, manifest.longitude,
    patch_size=CONFIG.patch_size, grid_crs=CONFIG.grid_crs,
    grid_spacing=CONFIG.grid_spacing, grid_tolerance=CONFIG.grid_tolerance)
print(json.dumps(grid.metadata, indent=2))
grid_locations = grid.location_frame(manifest.location.tolist())
display(grid_locations.groupby('complete_patch').agg(
    localita=('location', 'size'), celle_valide_min=('n_valid_cells', 'min'),
    celle_valide_max=('n_valid_cells', 'max')))

fig, ax = plt.subplots(figsize=(7, 6))
for complete, label, color in ((True, 'Finestra completa', 'tab:blue'),
                               (False, 'Finestra incompleta', 'tab:orange')):
    selected = grid_locations.complete_patch.eq(complete)
    ax.scatter(grid.column_indices[selected], grid.row_indices[selected],
               s=12, color=color, label=label)
ax.invert_yaxis()
ax.set(xlabel='Colonna della griglia', ylabel='Riga della griglia',
       title='Copertura delle finestre geografiche', aspect='equal')
ax.legend()
plt.show()

In [ ]:
counts = STGAN(n_features=3, hidden_size=CONFIG.hidden_size,
    n_layers=CONFIG.n_layers, cnn_channels=CONFIG.cnn_channels,
    cnn_layers=CONFIG.cnn_layers, patch_size=CONFIG.patch_size,
    kernel_size=CONFIG.kernel_size).parameter_counts()
parameters = pd.DataFrame({
    'baseline_GCN_GRU': {'generator': 63171, 'discriminator': 36769},
    'ConvGRU_con_maschera': counts,
})
parameters['differenza_pct'] = 100 * (
    parameters.ConvGRU_con_maschera / parameters.baseline_GCN_GRU - 1)
display(parameters)
print('Baseline: configurazione originale 3 variabili, hidden 64, 2 layer, 9 nodi.')

## Addestramento

La prima esecuzione avvia il training. Il protocollo completo visita tutte le
coppie località–timestamp: può richiedere molto tempo. Per una prova ridotta
impostare `train_samples_per_epoch` e usare un output distinto; lo scoring
resta sull'intero test. Un run completato con la stessa configurazione viene
letto senza ripetere il training. Una directory parziale non viene sovrascritta.

In [ ]:
import hashlib
from physiq_pv.anomaly_detection.stgan import ALIGNMENT_POLICY, STGANCNNConfig
existing_metadata = OUT_ROOT / 'run_metadata.json'
if existing_metadata.is_file():
    saved = json.loads(existing_metadata.read_text())
    saved_config = dict(saved['configuration']['model'])
    # Le precedenti run ConvGRU usavano sempre kernel 3x3.
    saved_config.setdefault('kernel_size', 3)
    # I campi runtime assenti nelle vecchie run assumono i default compatibili.
    for key in ('num_workers', 'persistent_workers', 'prefetch_factor', 'pin_memory', 'cache_normalized', 'shuffle_mode', 'shuffle_block_size', 'execution_mode', 'score_storage', 'score_memory_limit_mb', 'score_chunk_size'):
        saved_config.setdefault(key, getattr(STGANCNNConfig(), key))
    for key in ('train_num_workers', 'score_num_workers', 'score_batch_size', 'log_interval'):
        saved_config.setdefault(key, None)
    if (saved.get('alignment_policy') != ALIGNMENT_POLICY
        or saved_config != CONFIG.to_dict()
        or saved['paper_top_k_percent'] != TOP_K_PERCENT
        or saved['seeds'] != list(SEEDS)
        or saved.get('source_manifest_sha256') != hashlib.sha256(MANIFEST.read_bytes()).hexdigest()):
        raise ValueError('Il run salvato ha una architettura o configurazione diversa: scegliere un altro OUT_ROOT.')
    print('Run completato già presente:', OUT_ROOT)
elif RUN_TRAINING:
    run_stgan(manifest_path=MANIFEST, out_dir=OUT_ROOT,
        config=CONFIG, device=DEVICE, seeds=SEEDS,
        paper_top_k_percent=TOP_K_PERCENT)
else:
    print('Training disabilitato; le celle successive richiedono un run completato.')

In [ ]:
saved_run = json.loads((SEED_DIR.parent / 'run_metadata.json').read_text(encoding='utf-8'))
metadata = json.loads((SEED_DIR / 'metadata.json').read_text())
history = pd.read_csv(SEED_DIR / 'training_history.csv')
boundary = pd.read_csv(SEED_DIR / 'boundary_summary.csv')
display(boundary)
display(pd.Series(metadata['backend']['performance'], name='Prestazioni misurate'))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, column, label in zip(axes, ['generator_loss', 'discriminator_loss'],
                             ['Loss generatore', 'Loss discriminatore']):
    ax.plot(history.epoch, history[column], marker='o')
    ax.set(xlabel='Epoca', ylabel=label)
    ax.grid(alpha=.25)
fig.tight_layout()
figure_dir = SEED_DIR / 'figures'
figure_dir.mkdir(exist_ok=True)
fig.savefig(figure_dir / 'training_losses.png', dpi=160)
plt.show()

saved_config = dict(saved_run['configuration']['model'])
saved_config.setdefault('kernel_size', 3)
for key in ('num_workers', 'persistent_workers', 'prefetch_factor', 'pin_memory', 'cache_normalized', 'shuffle_mode', 'shuffle_block_size', 'execution_mode', 'score_storage', 'score_memory_limit_mb', 'score_chunk_size'):
    saved_config.setdefault(key, getattr(STGANCNNConfig(), key))
for key in ('train_num_workers', 'score_num_workers', 'score_batch_size', 'log_interval'):
    saved_config.setdefault(key, None)
summary_text = 'BEGIN_STGAN_CNN_SUMMARY\n' + json.dumps({
    'architecture': metadata['backend']['backend'],
    'configuration': saved_config,
    'parameters': metadata['backend']['parameter_counts'],
    'grid': metadata['backend']['grid'],
    'performance': metadata['backend']['performance'],
    'boundary_summary': boundary.to_dict(orient='records'),
    'decision_rule': 'global test top-K; prima del filtro di qualità posthoc',
    'top_k_percent': metadata['paper_top_k_percent'],
}, indent=2) + '\nEND_STGAN_CNN_SUMMARY\n'
print(summary_text)
(SEED_DIR / 'model_results_summary.txt').write_text(summary_text, encoding='utf-8')

## Confronto opzionale con la baseline

Confronta le decisioni salvate sulle stesse coppie località–timestamp, leggendo
una località alla volta. Riporta esplicitamente timestamp non abbinati.
Interni e bordi sono separati secondo la finestra CNN. Per interpretare gli
interni come pari contesto verificare prima l'audit di coincidenza con il kNN.

Si tratta di **accordo tra detector**, non di accuratezza rispetto a vere anomalie.
Questa cella usa i top-K esportati senza filtro di qualità o nuovo ranking.
Le tre analisi posthoc mantengono invece il loro protocollo clean top-1%.

In [ ]:
from physiq_pv.reporting.stgan_cnn_comparison import compare_stgan_exports
if (BASELINE_SEED_DIR / 'locations').is_dir():
    agreement, agreement_locations = compare_stgan_exports(
        SEED_DIR, BASELINE_SEED_DIR, out_dir=SEED_DIR / 'baseline_comparison')
    display(agreement)
else:
    print('Baseline non disponibile:', BASELINE_SEED_DIR)

## Ablation del kernel a input geografico fisso

La tabella confronta kernel **1x1, 3x3 e 5x5** con la stessa patch e tutte le
altre impostazioni identiche. Non avvia altri training e non sceglie un vincitore.
Per ogni esperimento cambiare solo `KERNEL_SIZE` nella configurazione e rieseguire
il notebook dall'inizio. La prima ablation consigliata e' 1x1 rispetto al 3x3 gia eseguito.

1x1 elimina lo scambio tra celle nei gate ConvGRU; le aggregazioni successive
restano presenti. 5x5 su input 3x3 usa piu padding senza aggiungere localita.
Il numero di parametri cambia: riportarlo insieme ai risultati.
Confrontare score, sovrapposizione delle anomalie e classifiche con gli stessi
filtri di qualita. Il numero globale di anomalie resta imposto dal top-1%.
Per verificare la stabilita, ripetere poi i confronti con gli stessi seed aggiuntivi.


In [ ]:
search_rows = []
for kernel in (1, 3, 5):
    candidate = replace(CONFIG, kernel_size=kernel)
    model = STGAN(n_features=3, hidden_size=candidate.hidden_size,
        n_layers=candidate.n_layers, cnn_channels=candidate.cnn_channels,
        cnn_layers=candidate.cnn_layers, patch_size=candidate.patch_size,
        kernel_size=candidate.kernel_size)
    search_rows.append({'kernel_size': kernel, 'patch_size': candidate.patch_size,
        'cnn_layers': candidate.cnn_layers, 'cnn_channels': candidate.cnn_channels,
        'hidden_size': candidate.hidden_size, **model.parameter_counts()})
search_plan = pd.DataFrame(search_rows)
display(search_plan)
search_plan.to_csv(SEED_DIR / 'candidate_configurations.csv', index=False)


## Riuso dei notebook di analisi

`stgan_pointwise_posthoc_sdenet.ipynb` legge di default la run kernel 3x3 e salva
una nuova sottocartella per ogni esecuzione sotto `outputs/sde_stgan_cnn_quality_filtered`.
Per una ablation selezionare esplicitamente `STGAN_SEED_DIR`: la cella seguente
stampa il codice da incollare prima della configurazione del posthoc.
`STGAN_CNN_OUT_DIR`, se impostata, prevale sul nome automatico della cartella;
un output esistente con configurazione diversa viene rifiutato.

Per personalizzare i percorsi o eseguire gli altri due notebook,
impostare le variabili prima di avviare Jupyter, oppure nelle celle di
configurazione dei tre notebook, usando directory CNN distinte:

- `STGAN_SEED_DIR`: directory `seed_20` di questo run;
- `STGAN_POSTHOC_ROOT`: `outputs/sde_stgan_cnn_quality_filtered`;
- `STGAN_INPUT_TARGET_OUT_DIR`: `outputs/stgan_cnn_input_target_cases_t1_t6`;
- `ANOMALY_SENSITIVITY_OUT_DIR`: `outputs/anomaly_threshold_sensitivity_cnn_t1_t6`.

Rieseguire `stgan_pointwise_posthoc_sdenet.ipynb`,
`stgan_input_target_cases_sdenet.ipynb` e
`anomaly_threshold_sensitivity_mtgflow_stgan.ipynb`.
Usare gli stessi dati di qualità e le stesse previsioni della baseline.
Questi notebook producono le analisi aggregate su tutte le località.
La sezione **Errori per tipo di estremo meteorologico** del
[notebook posthoc](stgan_pointwise_posthoc_sdenet.ipynb) mostra i boxplot MAE
t+1/t+6 e le bande RMSE t+1,...,t+6 per normali STGAN, temperatura estrema alta e bassa,
vento estremo forte e debole, irradianza estrema alta e irradianza estrema bassa.
Usa esclusivamente le previsioni **quality filtered**, con esclusione dei
dropout solari PVGIS e dell'ora di recovery. Le etichette meteorologiche
provengono dal CSV climatologico configurabile con `PVGIS_CLIMATOLOGY_SCORES`.
Le bande rappresentano il 25-75 percentile dell'RMSE fra localita; le
condizioni concomitanti possono appartenere a piu gruppi.


In [ ]:
print('Incollare prima della configurazione del notebook posthoc:')
print('import os')
print(f"os.environ['STGAN_SEED_DIR'] = {str(SEED_DIR)!r}")
